# Run-book 214-P5 — fused Sentinel-2 + Sentinel-1 training data on Azure ML

**What this proves:** spec 214 P5 (AC7). On Azure ML, `download(inputs=[s2, s1])` fills a blob archive,
and `create_training_data(inputs=[s2, s1], runner="aml")` builds one fused cube per `AT_2018_TRAIN` field in
`s2grid=4772924` for June 2018 (`mosaic_days="month"`, T = 1): 12 Sentinel-2 bands plus `vv` and `vh` on
Sentinel-2's B08 10 m grid. It also checks, one cell per phase, what P1–P4 and #227 delivered, so a failure
points at one phase. Sentinel-1 is never downloaded to this laptop: only the flattened arrays and one fused
cube (tens of MB) come home.

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs:** an Azure login that `~/.config/rise-eo/config.toml` describes (`rise_eo.config.load()`); an
Azure ML environment built from `main` **after this PR merges** (its name and version go in Settings); a blob
prefix of your own. Time (estimates): the download is minutes to tens of minutes (Sentinel-1 scenes are
~3.7 GB each, copied in-region, never to this laptop), the build is one AML job of a few minutes plus node
start-up, and the checks run in seconds. Laptop disk: tens of MB.

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `rise-eo/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with rise-eo's dependencies.
- Open VS Code on the **`rise-eo/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `rise-eo/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `rise-eo/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `rise-eo` gets imported.** rise-eo is installed from the *main* checkout, so inside a
worktree a plain `import rise_eo` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: downloads skip
files that are already complete. After a kernel restart, run **Setup** and **Settings** again,
then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which rise-eo. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be rise-eo's own virtual environment, rise-eo/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not rise-eo's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the rise-eo checkout that owns the venv

# 2. Which rise-eo? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise-eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("rise-eo    :", rise_eo.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else.

In [ ]:
# Settings
import datetime
import json
import re

import geopandas as gpd
import numpy as np
import rasterio.features

import rise_eo
from rise_eo.storage import fs
from rise_eo.storage.azure import configure_storage

cfg = rise_eo.config.load()      # subscription, resource group, workspace, cluster, uami: private, read at run time
configure_storage("azure")       # so every step can read blob, also after a kernel restart

# Replace <your-user> with your own blob prefix before running; a forgotten placeholder fails below.
AZ_ROOT = "abfss://data@strisewesteurope.dfs.core.windows.net/<your-user>/214-p5-fused"
assert "<" not in AZ_ROOT, "set your own blob prefix in AZ_ROOT (replace <your-user>)"
# The AML environment built from main after this PR merged: "<name>:<version>".
AZ_ENV = "<environment name>:<version>"
assert "<" not in AZ_ENV, "set AZ_ENV to the environment built from main (replace the placeholders)"

ARCHIVE = f"{AZ_ROOT}/imagery"           # one catalog per collection folder; Sentinel-1 stays on blob
RUN_ROOT = f"{AZ_ROOT}/run"              # blob working root: cubes, input.csv, the raw flatten output
RUN_FOLDER = f"{RUN_ROOT}/runs/train"    # where create_training_data puts the per-field cubes
OUT = MAIN_CHECKOUT / "tests" / "outputs" / "214_p5_fused"   # gitignored: the landed arrays and GeoTIFFs

SHAPES = CHECKOUT / "notebooks" / "shapefiles"
ROI = SHAPES / "s2grid=4772924.geojson"          # inside MGRS tile T33UWP, with labels
LABELS = SHAPES / "AT_2018_TRAIN.geojson"        # 900 EuroCrops fields, columns fid and crop
START = datetime.datetime(2018, 6, 1)            # one calendar month: T = 1
END = datetime.datetime(2018, 7, 1)
MOSAIC_DAYS = "month"
MAX_TILES = 40                                    # per collection; the download refuses a larger match

S2 = rise_eo.CollectionInput(
    "sentinel-2-l2a",
    ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12", "SCL"],
    processing="latest",                          # one processing per acquisition
)
S1 = rise_eo.CollectionInput(
    "sentinel-1-rtc", ["vv", "vh"],
    properties_filter={"sat:orbit_state": ["ascending", "descending"]},   # opt in to median both orbits (P1)
)
FIELD_ID = None                                   # the field whose fused cube Step 7 opens; None = first id

runner_kwargs = dict(
    cluster=cfg.cluster, environment=AZ_ENV, root=RUN_ROOT,
    identity_client_id=cfg.uami_client_id,        # the NODES' identity, not your login
    subscription_id=cfg.subscription_id, resource_group_name=cfg.resource_group,
    workspace_name=cfg.workspace, poll_interval_seconds=10,
)
OUT.mkdir(parents=True, exist_ok=True)

## Helpers

Three small functions the checks below share. Run this cell after **Settings** on every restart.

In [ ]:
# Helpers
def run_segments():
    """`{name: segment}` of the fused, Sentinel-2 and Sentinel-1 folders in the run folder on blob."""
    found = [p.rstrip("/").split("/")[-1] for p in fs.ls(RUN_FOLDER)]
    out = {}
    for name, pattern in (("fused", r"s2l2a-s1rtc_20180601-20180701_P1M_[0-9a-f]{8}"),
                          ("s2", r"s2l2a_20180601-20180701_P1M_[0-9a-f]{8}"),
                          ("s1", r"s1rtc-on-s2l2a_20180601-20180701_P1M_[0-9a-f]{8}")):
        hits = [s for s in found if re.fullmatch(pattern, s)]
        assert len(hits) == 1, f"expected one segment matching {pattern}, found {hits} in {found}"
        out[name] = hits[0]
    return out


def landed(name):
    """One landed training array, read from this laptop."""
    return np.load(OUT / "training" / f"{name}.npy")


def cube(segment, field):
    """`(datacube, metadata)` of `field`'s cube under `segment` on blob."""
    folder = f"{RUN_FOLDER}/{segment}/{field}"
    return (fs.load_npy(f"{folder}/datacube.npy"),
            fs.load_npy(f"{folder}/metadata.pickle.npy", allow_pickle=True)[()])

## Step 1 — download both collections to the blob archive (P4, AC4)

**What it does:** `rise_eo.download(inputs=[S2, S1], runner="aml")` runs one AML download per collection into
`ARCHIVE`, after both inputs pass preflight. It returns `{collection: catalog_filepath}`. Sentinel-1 scenes are
whole ~250 km RTC scenes copied blob to blob.
**PASS if:** the call returns both collections' catalogs, and both exist on blob.
**Time / disk:** minutes to tens of minutes on the cluster; nothing on this laptop. `MAX_TILES` makes the call
refuse, before any byte moves, when the ROI matches more granules than you expect.

In [ ]:
# Step 1 -- download.
catalogs = rise_eo.download(
    roi=str(ROI), startdate=START, enddate=END, bands=None, dst_folderpath=ARCHIVE,
    inputs=[S2, S1], source="mpc", max_tiles=MAX_TILES, storage="azure",
    runner="aml", runner_kwargs=runner_kwargs,
)
print(json.dumps(catalogs, indent=2))

In [ ]:
# Step 1 -- check it (P4): one catalog per collection, at the archive layout.
for collection in (S2.collection, S1.collection):
    expected = rise_eo.archive_catalog_filepath(ARCHIVE, collection)
    assert fs.exists(expected), f"no catalog for {collection} at {expected}"
print("both catalogs are on blob")

## Step 2 — Sentinel-1 holds both orbit states (P1, AC3)

**What it does:** reads the Sentinel-1 catalog and lists the `sat:orbit_state` of each granule.
**PASS if:** both `ascending` and `descending` are present. Step 4's build lists both orbit states in
`properties_filter`; without that opt-in the build would raise on mixed orbits, so a green build also shows the
opt-in reached the nodes.

In [ ]:
# Step 2 -- check it (P1).
s1_rows = fs.read_parquet(rise_eo.archive_catalog_filepath(ARCHIVE, S1.collection))
orbits = sorted({json.loads(p)["sat:orbit_state"] for p in s1_rows["properties"]})
print("Sentinel-1 granules:", len(s1_rows), "| orbit states:", orbits)
assert orbits == ["ascending", "descending"], f"expected both orbit states, found {orbits}"

## Step 3 — build the fused cubes and flatten them (P5, AC7)

**What it does:** `rise_eo.create_training_data(inputs=[S2, S1], archive_root=ARCHIVE, runner="aml")` over the
`AT_2018_TRAIN` fields inside `s2grid=4772924`. Each field gets a Sentinel-2 cube, a Sentinel-1 cube built onto
Sentinel-2's B08 grid, and the joined fused cube. The flatten leg lands the arrays in `OUT / "training"`.
**PASS if:** the call returns; Steps 4–9 hold.
**Time / disk:** one AML build fan-out plus a flatten job, then a transfer of the arrays (tens of MB).

In [ ]:
# Step 3 -- build and flatten.
fields = gpd.read_file(LABELS)
cell = gpd.read_file(ROI)
fields = fields.to_crs(cell.crs)
fields = fields[fields.intersects(cell.geometry.iloc[0])]     # the fields inside the cell
print(f"{len(fields)} fields inside {ROI.name}")

training = rise_eo.create_training_data(
    label_polygons=fields, catalog_filepath=None, startdate=START, enddate=END,
    mosaic_days=MOSAIC_DAYS, bands=None, id_col="fid", label_col="crop",
    inputs=[S2, S1], archive_root=ARCHIVE,
    export_folderpath=str(OUT / "training"), run_folderpath=RUN_FOLDER,
    runner="aml", runner_kwargs=runner_kwargs,
)
print(training.export_folderpath, training.n_pixels, "pixels, bands", training.bands)

## Step 4 — short, readable paths and `params.json` (P2, AC9)

**What it does:** lists the run folder on blob.
**PASS if:** it holds exactly one fused segment `s2l2a-s1rtc_20180601-20180701_P1M_<hash8>`, the Sentinel-2
component `s2l2a_20180601-20180701_P1M_<hash8>` and the Sentinel-1 component built on its grid
`s1rtc-on-s2l2a_20180601-20180701_P1M_<hash8>`, and the fused segment's `params.json` lists both components.

In [ ]:
# Step 4 -- check it (P2).
seg = run_segments()
print(seg)
with fs.open(f"{RUN_FOLDER}/{seg['fused']}/params.json", "r") as f:
    params = json.load(f)
assert [c["segment"] for c in params["components"]] == [seg["s2"], seg["s1"]], params["components"]
assert [c["collection"] for c in params["components"]] == [S2.collection, S1.collection]
print("segments and params.json are as D7 says")

## Step 5 — per-sample window starts (P3, AC2)

**What it does:** reads `window_starts.npy` and `ids.npy` from the landed arrays.
**PASS if:** `window_starts.npy` has one entry per sample (per row of `ids.npy`), each the window start
2018-06-01.

In [ ]:
# Step 5 -- check it (P3).
ids = landed("ids")
starts = landed("window_starts")
assert len(starts) == len(ids), f"{len(starts)} window starts for {len(ids)} samples"
assert (starts == np.datetime64("2018-06-01", "ns")).all(), np.unique(starts)
print(len(ids), "samples, every window start is 2018-06-01")

## Step 6 — ids stay text (#227)

**What it does:** compares `ids.npy` with the label file's `fid` column.
**PASS if:** `ids.npy` is a text array and holds exactly the label file's `fid` values (as text) for the fields
inside the cell.

In [ ]:
# Step 6 -- check it (#227).
ids = landed("ids")
cell = gpd.read_file(ROI)
fields = gpd.read_file(LABELS).to_crs(cell.crs)
fields = fields[fields.intersects(cell.geometry.iloc[0])]
assert ids.dtype.kind == "U", f"ids.npy is {ids.dtype}, not text"
assert set(np.unique(ids)) == set(fields["fid"].astype(str)), (
    sorted(set(np.unique(ids)) ^ set(fields["fid"].astype(str)))[:10])
print(len(np.unique(ids)), "ids, all equal to the label file's fid as text")

## Step 7 — the fused cube (P5, AC7)

**What it does:** loads one field's fused cube from blob (tens of MB) and its two component cubes.
**PASS if:** the fused cube is `(1, H, W, 14)` and `float32`, its nodata is 0 and its geotiff metadata says
`float32`; its bands are the 12 Sentinel-2 bands (SCL is dropped after masking) then `vv`, `vh`, and
`band_collections` maps each band to its collection in that order. The Sentinel-1 component cube has nodata
−32768, and the count of its valid pixels equal to 0.0 (which become nodata in the fused cube) is printed.

In [ ]:
# Step 7 -- check it (P5).
seg = run_segments()
FIELD = str(FIELD_ID) if FIELD_ID is not None else str(landed("ids")[0])
fused, fused_md = cube(seg["fused"], FIELD)
s1_cube, s1_md = cube(seg["s1"], FIELD)
S2_BANDS_OUT = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
assert fused.ndim == 4 and fused.shape[0] == 1 and fused.shape[-1] == 14, fused.shape
assert fused.dtype == np.float32, fused.dtype
assert fused_md["bands"] == S2_BANDS_OUT + ["vv", "vh"], fused_md["bands"]
assert fused_md["nodata"] == 0 and fused_md["geotiff_metadata"]["nodata"] == 0
assert fused_md["geotiff_metadata"]["dtype"] == "float32", fused_md["geotiff_metadata"]["dtype"]
assert list(fused_md["band_collections"].items()) == (
    [(b, S2.collection) for b in S2_BANDS_OUT] + [("vv", S1.collection), ("vh", S1.collection)])
assert s1_md["nodata"] == -32768, s1_md["nodata"]
valid_zero = int(((s1_cube == 0.0)).sum())
print(f"field {FIELD}: fused cube {fused.shape} {fused.dtype}")
print("valid Sentinel-1 pixels equal to 0.0 (nodata in the fused cube):", valid_zero,
      "of", int((s1_cube != -32768).sum()), "valid values")

## Step 8 — the Sentinel-1 footprint equals the field (A2)

**What it does:** rasterizes the field on the fused cube's grid (`all_touched=True`, as the Sentinel-2 crop
does) and reads the Sentinel-1 component cube, which was read with the field buffered by two pixels.
**PASS if:** every Sentinel-1 pixel outside the rasterized field is nodata, and some pixels inside are valid.

In [ ]:
# Step 8 -- check it (A2).
seg = run_segments()
FIELD = str(FIELD_ID) if FIELD_ID is not None else str(landed("ids")[0])
fused, fused_md = cube(seg["fused"], FIELD)
s1_cube, _ = cube(seg["s1"], FIELD)
gt = fused_md["geotiff_metadata"]
shapes = gpd.GeoSeries(fused_md["geometry"]["shape"], crs=fused_md["geometry"]["crs"]).to_crs(gt["crs"])
outside = rasterio.features.geometry_mask(shapes, out_shape=(gt["height"], gt["width"]),
                                          transform=gt["transform"], all_touched=True)
s1_valid = (s1_cube[0] != -32768).any(axis=-1)
assert not s1_valid[outside].any(), f"{int(s1_valid[outside].sum())} valid Sentinel-1 pixels outside the field"
assert s1_valid[~outside].any(), "no valid Sentinel-1 pixel inside the field"
print(f"{int(s1_valid.sum())} valid Sentinel-1 pixels, all inside the field's {int((~outside).sum())} pixels")

## Step 9 — write `B08` and `vv` GeoTIFFs, then check the edges in QGIS

**What it does:** writes the fused cube's `B08` and `vv` as one-band GeoTIFFs under `OUT`.
**PASS if:** the two files exist. Then open both in QGIS: the field edges in `vv` line up with the edges in
`B08` (this is the visual check no assert can make).

In [ ]:
# Step 9 -- write the two GeoTIFFs.
from rise_eo.raster.images import save_geotiff

seg = run_segments()
FIELD = str(FIELD_ID) if FIELD_ID is not None else str(landed("ids")[0])
fused, fused_md = cube(seg["fused"], FIELD)
gt = fused_md["geotiff_metadata"]

for band in ("B08", "vv"):
    index = fused_md["bands"].index(band)
    save_geotiff(str(OUT / f"{band}.tif"), fused[0, :, :, index][None], {**gt, "count": 1})
    assert (OUT / f"{band}.tif").stat().st_size > 0
print("open in QGIS:", OUT / "B08.tif", OUT / "vv.tif")

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart.

In [ ]:
seg = run_segments()
FIELD = str(FIELD_ID) if FIELD_ID is not None else str(landed("ids")[0])
fused, fused_md = cube(seg["fused"], FIELD)
s1_cube, _ = cube(seg["s1"], FIELD)
ids = landed("ids")
summary = {
    "fields_inside_cell": int(len(np.unique(ids))),
    "samples": int(len(ids)),
    "segments": seg,
    "fused_cube_shape": list(fused.shape),
    "fused_dtype": str(fused.dtype),
    "fused_bands": fused_md["bands"],
    "valid_s1_pixels_equal_to_0.0": int((s1_cube == 0.0).sum()),
    "geotiffs": [str(OUT / "B08.tif"), str(OUT / "vv.tif")],
}
print(json.dumps(summary, indent=2, default=str))